# Session 5 — Full AI Pipeline
## Empire & Ink: AI Mughal Art Studio

**What you'll build:** An end-to-end orchestrator that takes a text prompt or uploaded photo, transforms it to Mughal art, stores the image in Supabase Storage, and saves metadata to the database — all in one Python function call.

---

## Setup

In [ ]:
!pip install google-genai supabase pillow python-dotenv

## Theory 1 — The Orchestrator Pattern

An **orchestrator** coordinates multiple services — like a conductor. It doesn't do the work itself; it calls specialists in sequence.

```
run_generate_pipeline(prompt, style)
  ├── enhance_prompt()       → enhanced text
  ├── generate_image()       → image bytes
  ├── upload_image()         → public URL
  └── save_gallery_item()    → DB row
```

Each step is independently testable. If one fails, the orchestrator returns a clean error.

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class GenerationResult:
    success: bool
    image_url: Optional[str] = None
    enhanced_prompt: Optional[str] = None
    gallery_id: Optional[str] = None
    error: Optional[str] = None

ok   = GenerationResult(success=True, image_url="https://...", gallery_id="abc-123")
fail = GenerationResult(success=False, error="Quota exceeded")
print("Success:", ok)
print("Failure:", fail)

## Theory 2 — Multimodal AI

Gemini Flash accepts both **text AND images** as input. This lets us build an image transformer:

```
User photo (bytes) + "Transform to Mughal miniature" → Gemini Flash → Mughal art image
```

We send the original image as base64-encoded bytes alongside the text prompt.

In [ ]:
import base64

def encode_image_for_api(image_bytes: bytes) -> str:
    return base64.b64encode(image_bytes).decode("utf-8")

fake_bytes = b"\xff\xd8\xff\xe0"
encoded = encode_image_for_api(fake_bytes)
print(f"Encoded {len(fake_bytes)} bytes -> {len(encoded)} base64 chars")
print(f"Ratio: {len(encoded)/len(fake_bytes):.1f}x (base64 is ~33% larger)")

## Theory 3 — Supabase Storage Upload

Supabase Storage works like a cloud file system:
- **Bucket**: top-level folder (e.g. `gallery-images`)
- **Path**: file location within bucket (e.g. `user123/2025-01-01-abc.png`)
- **Public URL**: CDN link anyone can load in a browser

```python
supabase.storage.from_("gallery-images").upload(
    path="user123/image.png",
    file=image_bytes,
    file_options={"content-type": "image/png"}
)
```

In [ ]:
import uuid
from datetime import datetime

def build_storage_path(user_id: str) -> str:
    timestamp = datetime.utcnow().strftime("%Y%m%d-%H%M%S")
    unique_id = str(uuid.uuid4())[:8]
    return f"{user_id}/{timestamp}-{unique_id}.png"

path = build_storage_path("user-abc123")
print("Storage path:", path)

---
## In-Class Exercises

### Exercise 1 — transform_image() using Gemini Flash multimodal

In [ ]:
import os
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    SUPABASE_URL   = userdata.get("SUPABASE_URL")
    SUPABASE_KEY   = userdata.get("SUPABASE_KEY")
except Exception:
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
    SUPABASE_URL   = os.getenv("SUPABASE_URL", "")
    SUPABASE_KEY   = os.getenv("SUPABASE_KEY", "")

def transform_image(image_bytes: bytes, style: str = "akbari") -> bytes:
    if not GEMINI_API_KEY:
        print("No API key — returning original bytes as placeholder")
        return image_bytes
    from google import genai
    client = genai.Client(api_key=GEMINI_API_KEY)
    # YOUR CODE HERE
    # Use client.models.generate_content() with image bytes + text prompt
    # Return the transformed image bytes
    return image_bytes  # placeholder

print("transform_image defined")

### Exercise 2 — upload_image() to Supabase Storage

In [ ]:
def upload_image(supabase_client, image_bytes: bytes, user_id: str,
                 bucket: str = "gallery-images") -> str:
    path = build_storage_path(user_id)
    # YOUR CODE HERE
    # supabase_client.storage.from_(bucket).upload(path, image_bytes,
    #     file_options={"content-type": "image/png"})
    # return supabase_client.storage.from_(bucket).get_public_url(path)
    return f"https://example.com/storage/{path}"  # placeholder

print("upload_image:", upload_image(None, b"bytes", "user1"))

### Exercise 3 — run_generate_pipeline() orchestrator

In [ ]:
def run_generate_pipeline(supabase_client, user_id: str,
                          prompt: str, style: str = "akbari") -> GenerationResult:
    try:
        # YOUR CODE HERE
        # Step 1: enhance_prompt(prompt, style)
        # Step 2: generate_image() -> bytes
        # Step 3: upload_image() -> url
        # Step 4: save_gallery_item() -> db row
        # Step 5: return GenerationResult(success=True, ...)
        return GenerationResult(
            success=True,
            image_url="https://placeholder.com/image.png",
            enhanced_prompt=f"[Enhanced] {prompt}",
            gallery_id="placeholder-id",
        )
    except Exception as e:
        return GenerationResult(success=False, error=str(e))

result = run_generate_pipeline(None, "user1", "A peacock in a garden")
print("Result:", result)

### Exercise 4 — Full end-to-end test

In [ ]:
if GEMINI_API_KEY and SUPABASE_URL and SUPABASE_KEY:
    from supabase import create_client
    sb = create_client(SUPABASE_URL, SUPABASE_KEY)
    r = run_generate_pipeline(sb, "test-user", "A white peacock on a marble terrace", "shahjahani")
    print("Real result:", r)
else:
    r = run_generate_pipeline(None, "test-user", "A white peacock", "shahjahani")
    print("Placeholder result:", r)
    print("\nAdd API keys as Colab secrets to run the real pipeline.")

---
## Post-Class Exercises

### Challenge 1 — GenerationResult with extra fields

In [ ]:
from dataclasses import dataclass, asdict
from typing import Optional

@dataclass
class GenerationResult:
    success: bool
    generation_type: str = "generate"
    image_url: Optional[str] = None
    thumbnail_url: Optional[str] = None
    enhanced_prompt: Optional[str] = None
    gallery_id: Optional[str] = None
    error: Optional[str] = None
    retry_count: int = 0

    # YOUR CODE HERE — add to_dict() that returns asdict(self)
    def to_dict(self):
        pass

r = GenerationResult(success=True, image_url="https://example.com/1.png")
print(r.to_dict())

### Challenge 2 — run_transform_pipeline()

In [ ]:
def run_transform_pipeline(supabase_client, user_id: str,
                           image_bytes: bytes, style: str = "akbari") -> GenerationResult:
    try:
        # YOUR CODE HERE
        # Step 1: transform_image(image_bytes, style) -> transformed_bytes
        # Step 2: upload_image() -> url
        # Step 3: save_gallery_item() with generation_type="transform"
        return GenerationResult(success=True, generation_type="transform",
                               image_url="https://placeholder.com/t.png")
    except Exception as e:
        return GenerationResult(success=False, error=str(e))

r = run_transform_pipeline(None, "user1", b"fake-bytes", "akbari")
print(r)

### Challenge 3 — Test transform with a downloaded sample image

In [ ]:
import urllib.request
from PIL import Image
import io

SAMPLE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1e/Gatto_europeo4.jpg/320px-Gatto_europeo4.jpg"
try:
    with urllib.request.urlopen(SAMPLE_URL) as resp:
        image_bytes = resp.read()
    img = Image.open(io.BytesIO(image_bytes))
    print(f"Downloaded: {img.size} {img.mode}")
    result = run_transform_pipeline(None, "test-user", image_bytes, "jahangiri")
    print("Transform result:", result)
except Exception as e:
    print("Download failed:", e)

### Challenge 4 — Retry mechanism for quota errors

In [ ]:
import time

def retry_pipeline(pipeline_fn, *args, max_retries=3, delay=5):
    for attempt in range(max_retries + 1):
        result = pipeline_fn(*args)
        if result.success:
            result.retry_count = attempt
            return result
        if result.error and "quota" in result.error.lower() and attempt < max_retries:
            print(f"Quota error, retrying in {delay}s (attempt {attempt+1}/{max_retries})...")
            time.sleep(delay)
        else:
            result.retry_count = attempt
            return result
    return result

def always_fails(user_id, prompt):
    return GenerationResult(success=False, error="quota exceeded")

r = retry_pipeline(always_fails, "user1", "a tiger", max_retries=2, delay=0)
print("Final after retries:", r)

---
## What you built today

- Orchestrated a multi-step AI pipeline: enhance → generate → upload → save
- Used Gemini Flash multimodal to transform photos into Mughal art
- Uploaded images to Supabase Storage and retrieved public CDN URLs
- Built retry logic to handle API quota errors gracefully

**Next session:** Session 6 — Auth & Security — protect your API so only logged-in users can generate art!